# real_groundtruth_downscale.ipynb

Reference validation notebook for the J-STARS paper:

**healpix-resample: a PSF-Aware HEALPix Resampling for Earth Observation Imagery**

## Why this notebook exists

The paper's two other real-data-adjacent checks each have a specific limit:

- The **UTM -> HEALPix -> UTM round trip** (`test-resample-paper.ipynb`) has no
  unblurred reference at all. It measures self-consistency with the input,
  not recovery of anything the method didn't already see.
- The **controlled synthetic-detector experiment** (`test-resample-paper.ipynb`,
  Esri World Imagery) has a real ground truth, but a *synthetic* one: the
  observation is generated by blurring that ground truth with the exact same
  Gaussian kernel the reconstruction operator then assumes. Recovering it is
  not, by itself, evidence that the method works on a real sensor's actual
  (non-Gaussian, unknown) response.

This notebook closes that gap using **only real Sentinel-2 data**: the real
10 m B04 band serves as both the source of a synthetic degradation *and* the
evaluation ground truth. Concretely, per scene:

1. Take the real, native 10 m Sentinel-2 B04 patch (`truth10`).
2. Degrade it to a synthetic 20 m observation (`degraded20`) using a
   deliberately **non-Gaussian** degradation pipeline (Gaussian blur +
   box-average + downsample), so the reconstruction operator's assumed
   Gaussian response does not exactly match the process that generated the
   data it is inverting.
3. Reconstruct a 10 m-equivalent field from `degraded20` with the PSF-aware
   method (and with nearest/bilinear/bicubic/Richardson-Lucy baselines).
4. Compare every reconstruction directly against `truth10` -- a real,
   unsynthesized Sentinel-2 image, not a texture the method has any special
   relationship to.

This is the real-data reconstruction-quality test the paper's Discussion and
Appendix (Section on generalization) point to as future work. It reuses the
same four benchmark scenes, the same cached patches, and the same
`healpix-resample` API as `test-resample-paper.ipynb`.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import torch
import xarray as xr
import pyproj
import pystac_client
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter
from scipy.interpolate import RegularGridInterpolator
from sklearn.metrics import mean_squared_error, mean_absolute_error
from skimage.metrics import structural_similarity as ssim

try:
    from healpix_resample import PSFResampler, BicubicResampler, fwhm_to_scale, recommend_npt
except Exception as exc:
    PSFResampler = BicubicResampler = fwhm_to_scale = recommend_npt = None
    warnings.warn(f"Could not import healpix_resample: {exc}")

try:
    from skimage.restoration import richardson_lucy
except Exception as exc:
    richardson_lucy = None
    warnings.warn(f"Could not import richardson_lucy: {exc}")

# DATA_DIR is the SAME cache directory used by test-resample-paper.ipynb. If
# the four benchmark scenes were already fetched there, this notebook reuses
# those real Sentinel-2 patches instead of re-downloading them.
DATA_DIR  = Path("data")
FIG_DIR   = Path("figures")
TABLE_DIR = Path("tables")
for d in [DATA_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(exist_ok=True)

RANDOM_SEED = 1234
np.random.seed(RANDOM_SEED)

# -- Real data -----------------------------------------------------------------
MIRROR_BASE = ("https://grid4earth.s3.gra.io.cloud.ovh.net"
               "/public/eopf-mirror/sentinel-2-l2a/patches")
PATCH_SIZE   = 256          # native 10 m pixels per side (must stay even)
CENTRAL_SIZE = PATCH_SIZE
BAND         = "b04"        # Sentinel-2 red band, 10 m native
PIXEL_SIZE_M = 10.0
DEVICE       = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# -- Degradation model: real 10 m truth -> synthetic 20 m observation ----------
# The "20 m" image is NOT a second real sensor: Sentinel-2 B04 has no native
# 20 m counterpart to use as an independent check. It is produced here by
# degrading the real 10 m band, so it is only as realistic as this model.
# What it buys over the paper's other experiments is that the *evaluation
# reference* (truth10) is real, unsynthesized Sentinel-2 imagery, not a
# texture the reconstruction operator has any special relationship to.
#
# The degradation pipeline is deliberately NOT a single clean Gaussian
# point-kernel, to avoid an "inverse crime" (recovering data only because the
# reconstruction operator assumes exactly the process that generated it):
#   1. Gaussian blur at DEGRADE_FWHM_M (optical/atmospheric blur term)
#   2. block-average BLOCK x BLOCK pixels (detector footprint integration)
#      and subsample to the coarser pitch
# A Gaussian convolved with a boxcar is not itself Gaussian, so the
# reconstruction operator's assumed Gaussian response (below) is already a
# deliberate simplification relative to how the "observation" was built.
DEGRADE_FWHM_M = 14.0        # metres; optical/atmospheric blur term only
BLOCK          = 2           # 10 m -> 20 m
PIXEL_SIZE_20M = PIXEL_SIZE_M * BLOCK

# -- Reconstruction (PSF-aware method under test) -------------------------------
HEALPIX_LEVEL = 20           # same level used throughout the paper (~6.22 m cells)
RECON_FWHM_M  = 25.0         # assumed effective 20 m response (1.25x pixel pitch,
                              # matching the FWHM/pixel-pitch ratio used at 10 m
                              # elsewhere in the paper); deliberately not equal
                              # to the true degradation kernel above.
LAMBDA_20M    = 0.01
MAX_ITER      = 14
THRESHOLD     = 0.1
READOUT_NPT   = 16           # BicubicResampler default, used only to read the
                              # reconstructed field out at the true 10 m centres

# -- Richardson-Lucy baseline ----------------------------------------------------
RICHARDSON_LUCY_ITER = 100

# -- Evaluation -------------------------------------------------------------------
BORDER_CROP_PX = 20   # 10 m pixels excluded from every metric on each side, to
                        # avoid boundary artefacts from the Gaussian blur's
                        # reflect padding and from partially-supported HEALPix
                        # cells at the patch edge.
CLASSICAL_METHODS = ["nearest", "linear", "cubic"]
CLASSICAL_LABELS = {"nearest": "Nearest", "linear": "Bilinear", "cubic": "Bicubic"}


## 2. Benchmark scenes

Same four scenes, pinned to the same GRID4EARTH-mirrored products, as `test-resample-paper.ipynb` -- so this notebook draws on the same real Sentinel-2 patches used everywhere else in the paper.

In [ ]:
benchmark_coordinates = {
    "urban": {
        "location": "Paris, France",
        "wgs84": {"lat": 48.8566, "lon": 2.3522},
        "recommended_date": "2026-05-01/2026-05-30",
        "product_id": "S2B_MSIL2A_20260527T105619_N0512_R094_T31UDQ_20260527T133135",
        "cloud": 20,
    },
    "water": {
        "location": "Lake Geneva, Switzerland",
        "wgs84": {"lat": 46.4983, "lon": 6.6327},
        "recommended_date": "2026-06-01/2026-06-24",
        "product_id": "S2A_MSIL2A_20260624T103041_N0512_R108_T32TLS_20260624T182910",
        "cloud": 20,
    },
    "forest": {
        "location": "Black Forest, Germany",
        "wgs84": {"lat": 48.265, "lon": 8.016},
        "recommended_date": "2026-07-01/2026-07-31",
        "product_id": "S2A_MSIL2A_20260704T102701_N0512_R108_T32UMU_20260704T170718",
        "cloud": 20,
    },
    "agriculture": {
        "location": "Beauce plains, France",
        "wgs84": {"lat": 48.258, "lon": 1.499},
        "recommended_date": "2026-06-01/2026-06-30",
        "product_id": "S2B_MSIL2A_20260623T104619_N0512_R051_T31UCP_20260623T132606",
        "cloud": 20,
    },
}


## 3. Sentinel-2 fetch utilities

Self-contained copy of `test-resample-paper.ipynb`'s extraction logic, unchanged, so this notebook can run standalone. Because the cache filename convention (`DATA_DIR / f"{scene}_data.zarr"`) is identical, if that notebook already populated `data/`, nothing is re-downloaded here.

In [ ]:
def _crs_of(obj):
    """CRS of an EOPF product or a cached patch.

    `obj.crs_code` is not provided by xarray-eopf 0.3.0, and the metadata key
    is `horizontal_CRS_code` (capital CRS), not `horizontal_crs_code`. The CF
    grid-mapping coordinate is the stable place to look, so prefer it.
    """
    try:
        return pyproj.CRS.from_wkt(obj.spatial_ref.attrs["crs_wkt"])
    except Exception:
        try:
            meta = getattr(obj, "attrs", {}).get("other_metadata", {})
            return pyproj.CRS.from_user_input(meta["horizontal_CRS_code"])
        except Exception:
            meta = getattr(obj, "attrs", {}).get("other_metadata", {})
            return pyproj.CRS.from_user_input(meta["horizontal_crs_code"])


def _add_latlon(ds, transformer):
    xx, yy = np.meshgrid(ds.x.values, ds.y.values)
    lon, lat = transformer.transform(xx, yy)
    ds = ds.assign_coords(longitude=(["y", "x"], lon), latitude=(["y", "x"], lat))
    return ds


def extract_bench_data(scene, patch_size=PATCH_SIZE, force=False, coords=None, band=None):
    """Fetch (or load from cache) the native 10 m patch for one scene.

    `coords` defaults to `benchmark_coordinates[scene]`. Pass it explicitly to
    fetch a one-off location without registering it in `benchmark_coordinates`.
    """
    if band is None:
        band = BAND

    xr.set_options(keep_attrs=True, display_expand_attrs=False)
    catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")
    path = DATA_DIR / f"{scene}_data.zarr"
    if path.exists() and not force:
        try:
            import zarr as _zarr
            _a = _zarr.open_group(str(path), mode="r").attrs.asdict()
            if _a.get("source_item_id"):
                print(f"[{scene}] cached  <- {_a['source_item_id']}"
                      f"  ({_a.get('source_datetime', '?')},"
                      f" cloud {_a.get('source_cloud_cover', '?')}%)")
            else:
                print(f"[{scene}] cached  <- provenance not recorded "
                      f"(cache predates source_item_id; delete it to refresh)")
        except Exception as _exc:
            print(f"[{scene}] cached  <- could not read provenance: {_exc}")
        return
    coords = benchmark_coordinates[scene] if coords is None else coords
    lat0, lon0 = coords["wgs84"]["lat"], coords["wgs84"]["lon"]

    pinned = coords.get("product_id")
    if pinned:
        try:
            import fsspec
            _ds = xr.open_zarr(
                fsspec.get_mapper(f"{MIRROR_BASE}/{pinned}.zarr"), consolidated=True
            )
            _ds.to_zarr(path, mode="w")
            print(f"[{scene}] mirror  <- {pinned}"
                  f"  (cloud {_ds.attrs.get('source_cloud_cover', '?')}%)")
            return
        except Exception as exc:
            print(f"[{scene}] mirror unavailable ({type(exc).__name__}), "
                  f"falling back to the STAC search: {str(exc)[:70]}")

    items = list(catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[lon0 - 0.05, lat0 - 0.05, lon0 + 0.05, lat0 + 0.05],
        datetime=coords["recommended_date"],
        query={"eo:cloud_cover": {"lt": coords["cloud"]}},
    ).get_all_items())
    if not items:
        raise RuntimeError(f"No items for {scene}")
    item = items[0]

    _props = item.properties
    print(f"[{scene}] {len(items)} product(s) matched; using {item.id}"
          f"  ({_props.get('datetime', '?')},"
          f" cloud {_props.get('eo:cloud_cover', '?')}%)")
    if len(items) > 1:
        print(f"[{scene}]   others: {', '.join(i.id for i in items[1:4])}"
              f"{' ...' if len(items) > 4 else ''}")

    ds = xr.open_dataset(
        item.assets["product"].href,
        engine="eopf-zarr",
        resolution=10,
        variables=[band],
        chunks={},
    )

    try:
        src_crs = pyproj.CRS.from_wkt(ds.spatial_ref.attrs["crs_wkt"])
    except Exception:
        try:
            src_crs = pyproj.CRS.from_user_input(
                ds.attrs["other_metadata"]["horizontal_CRS_code"]
            )
        except Exception:
            src_crs = pyproj.CRS.from_user_input(
                ds.attrs["other_metadata"]["horizontal_crs_code"]
            )
    transformer = pyproj.Transformer.from_crs(
        src_crs, pyproj.CRS.from_epsg(4326), always_xy=True
    )

    _fwd = pyproj.Transformer.from_crs(
        pyproj.CRS.from_epsg(4326), _crs_of(ds), always_xy=True
    )
    x_c, y_c = _fwd.transform(lon0, lat0)
    half = patch_size // 2 * 10
    ds_patch = ds.sel(
        x=slice(x_c - half, x_c + half),
        y=slice(y_c + half, y_c - half),
    )
    ds_patch = _add_latlon(ds_patch, transformer)
    ds_patch.attrs["source_item_id"] = item.id
    ds_patch.attrs["source_collection"] = "sentinel-2-l2a"
    ds_patch.attrs["source_stac_endpoint"] = "https://stac.core.eopf.eodc.eu"
    ds_patch.attrs["source_datetime"] = str(_props.get("datetime", ""))
    ds_patch.attrs["source_cloud_cover"] = str(_props.get("eo:cloud_cover", ""))
    ds_patch.to_zarr(path, mode="w")


## 4. Download or load scenes (real, native 10 m)

In [ ]:
FORCE_DOWNLOAD = False
for scene in benchmark_coordinates:
    try:
        extract_bench_data(scene, force=FORCE_DOWNLOAD)
        print(f"[OK] {scene}")
    except Exception as exc:
        print(f"[WARN] {scene}: {exc}")


## 5. Common utilities: loading, UTM axes, block-reduce

In [ ]:
def fill_nan_with_mean(img):
    img = np.asarray(img, dtype=np.float32).copy()
    mask = np.isfinite(img)
    if not mask.any():
        raise ValueError("Image contains no finite pixels.")
    img[~mask] = np.nanmean(img)
    return img


def load_scene_patch(scene_name):
    path = DATA_DIR / f"{scene_name}_data.zarr"
    dt = xr.open_datatree(path, engine="zarr", consolidated=False, chunks={})
    da = dt[BAND]
    img = fill_nan_with_mean(da.values)
    lon = da["longitude"].values.astype(np.float64)
    lat = da["latitude"].values.astype(np.float64)
    return img, lon, lat, da


def central_crop(img, size=CENTRAL_SIZE):
    ny, nx = img.shape
    cy, cx = ny // 2, nx // 2
    h = size // 2
    return img[cy - h:cy + h, cx - h:cx + h]


def get_utm_axes(da, central_size=CENTRAL_SIZE):
    """1-D UTM x and y axes for the central crop of a DataArray."""
    x_full = da.x.values
    y_full = da.y.values
    nx, ny = x_full.size, y_full.size
    cx, cy = nx // 2, ny // 2
    h = central_size // 2
    x0 = x_full[cx - h:cx + h]
    y0 = y_full[cy - h:cy + h]
    return x0, y0


def block_reduce_mean(arr, block=BLOCK):
    """Average non-overlapping block x block tiles. Requires both dimensions
    to be exact multiples of `block` (true for PATCH_SIZE=256, block=2)."""
    ny, nx = arr.shape
    assert ny % block == 0 and nx % block == 0, "array shape must be a multiple of `block`"
    return arr.reshape(ny // block, block, nx // block, block).mean(axis=(1, 3))


def crop_border(img, border=BORDER_CROP_PX):
    if border <= 0:
        return img
    return img[border:-border, border:-border]


## 6. Degradation model: real 10 m truth -> synthetic 20 m observation

See the configuration cell for the rationale. `build_20m_grid` returns the real 10 m truth, its geolocation, and the corresponding synthetic 20 m observation with matching geolocation, all derived from a *single* real cached patch.

In [ ]:
def _fwhm_to_gauss_sigma_px(fwhm_m, pixel_size_m=PIXEL_SIZE_M):
    """Standard Gaussian FWHM -> sigma (pixels), for scipy.ndimage.gaussian_filter.

    NOTE: this is the ordinary sigma = FWHM / (2*sqrt(2 ln 2)) convention --
    NOT the healpix_resample package's own `s = FWHM / sqrt(2 ln 2)` scale
    (see healpix_resample.psf_geometry's module docstring). The two are
    unrelated conventions used in two different places in this notebook and
    must not be swapped: this one feeds scipy's blur, `fwhm_to_scale()`
    (imported from healpix_resample) feeds the PSFResampler kernel below.
    """
    sigma_m = fwhm_m / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    return sigma_m / pixel_size_m


def degrade_to_20m(img10, fwhm_m=DEGRADE_FWHM_M, block=BLOCK):
    """Real 10 m Sentinel-2 B04 -> synthetic 20 m observation.

    Gaussian blur (optical/atmospheric term) followed by block-averaging
    (detector-footprint integration + downsampling). A Gaussian convolved
    with a boxcar is not itself Gaussian, so this is a genuinely different
    functional form from the pure Gaussian point-kernel the reconstruction
    operator assumes -- recovering `truth10` is therefore not a matter of
    inverting the exact process that generated `degraded20`.
    """
    sigma_px = _fwhm_to_gauss_sigma_px(fwhm_m)
    blurred = gaussian_filter(img10, sigma=sigma_px, mode="reflect")
    return block_reduce_mean(blurred, block=block)


def build_20m_grid(scene_name):
    """Real 10 m truth + geolocation, and the synthetic 20 m observation +
    geolocation, for one scene."""
    img10, lon10, lat10, da = load_scene_patch(scene_name)
    img10 = central_crop(img10)
    lon10 = central_crop(lon10)
    lat10 = central_crop(lat10)

    img20 = degrade_to_20m(img10)
    lon20 = block_reduce_mean(lon10, block=BLOCK)
    lat20 = block_reduce_mean(lat10, block=BLOCK)

    x10, y10 = get_utm_axes(da)
    x20 = 0.5 * (x10[0::2] + x10[1::2])
    y20 = 0.5 * (y10[0::2] + y10[1::2])

    return dict(img10=img10, lon10=lon10, lat10=lat10, x10=x10, y10=y10,
                img20=img20, lon20=lon20, lat20=lat20, x20=x20, y20=y20, da=da)


## 7. PSF-aware reconstruction (matched effective response)

`PSFResampler` is built from the 20 m degraded samples and solves for a HEALPix field at level 20. To evaluate against the real 10 m grid -- a different, denser set of points than the operator's own 20 m samples -- `sample_healpix_field_at` builds a *second*, purely-interpolating resampler (`BicubicResampler`) whose sample points are the true 10 m centres and whose output cell space is restricted to the first operator's retained cells via `out_cell_ids`. Calling `.invert()` on that second instance reads the reconstructed field out at the 10 m centres with no further deconvolution -- exactly the composition `out_cell_ids` is designed for in this package.

In [ ]:
def sample_healpix_field_at(cell_ids_src, cell_data_src, lon_q, lat_q, level,
                             device=DEVICE, npt=READOUT_NPT, threshold=THRESHOLD):
    """Interpolate a HEALPix cell-space field at arbitrary query points.

    `cell_ids_src`/`cell_data_src` is the (cell_ids, cell_data) pair returned
    by a PSFResampler's `.resample()`, defined only on the sparse set of
    HEALPix cells that operator retained. To read the reconstructed field out
    at *different* points than the operator's own samples, build a second
    resampler whose own sample points are the query points and whose output
    cell space is restricted to `cell_ids_src` via `out_cell_ids`, then call
    `.invert()` on it -- `.invert()` uses that instance's own adjoint operator
    MT, i.e. a bicubic-kernel interpolation of a HEALPix-cell field at its
    sample points, which is exactly what is needed here.
    """
    readout = BicubicResampler(
        lon_deg=np.asarray(lon_q).reshape(-1),
        lat_deg=np.asarray(lat_q).reshape(-1),
        level=level, out_cell_ids=cell_ids_src, Npt=npt,
        threshold=threshold, device=device, verbose=False,
    )
    dest_ids = readout.get_cell_ids()
    pos = {int(c): i for i, c in enumerate(np.asarray(cell_ids_src))}
    sel = np.array([pos[int(c)] for c in dest_ids], dtype=np.int64)
    aligned = np.asarray(cell_data_src)[sel]
    out = readout.invert(aligned)
    return np.asarray(out)


def reconstruct_psf_aware(scene_name, force=False):
    out_npz = DATA_DIR / f"{scene_name}_real_downscale_psf_aware.npz"
    if out_npz.exists() and not force:
        return dict(np.load(out_npz, allow_pickle=True))

    if PSFResampler is None:
        raise ImportError("PSFResampler is not available. Install healpix-resample.")

    g = build_20m_grid(scene_name)

    scale_m = fwhm_to_scale(RECON_FWHM_M)
    npt = recommend_npt(scale_m, HEALPIX_LEVEL)["npt"]

    t0 = time.time()
    resampler = PSFResampler(
        lon_deg=g["lon20"].reshape(-1), lat_deg=g["lat20"].reshape(-1),
        level=HEALPIX_LEVEL, sigma_m=scale_m, Npt=npt,
        threshold=THRESHOLD, device=DEVICE, verbose=False,
    )
    build_time = time.time() - t0

    t1 = time.time()
    res = resampler.resample(g["img20"].reshape(-1), lam=LAMBDA_20M, max_iter=MAX_ITER)
    solve_time = time.time() - t1

    t2 = time.time()
    rec_flat = sample_healpix_field_at(
        res.cell_ids, res.cell_data, g["lon10"], g["lat10"], HEALPIX_LEVEL,
    )
    readout_time = time.time() - t2

    rec10 = np.asarray(rec_flat).reshape(g["img10"].shape)

    np.savez_compressed(
        out_npz, scene=scene_name,
        truth10=g["img10"].astype(np.float32),
        degraded20=g["img20"].astype(np.float32),
        reconstructed10=rec10.astype(np.float32),
        build_time=build_time, solve_time=solve_time, readout_time=readout_time,
    )
    return dict(np.load(out_npz, allow_pickle=True))


## 8. Classical geometric baselines (nearest / bilinear / bicubic)

Purely geometric upsampling of the 20 m observation onto the true 10 m grid, with no spatial-response model -- the same role these baselines play everywhere else in the paper.

In [ ]:
def _axes_ascending(y_ax, x_ax, img):
    """RegularGridInterpolator requires ascending axes; UTM y is usually
    descending (north-up raster). Flip whichever axis needs it."""
    if y_ax[0] > y_ax[-1]:
        y_ax, img = y_ax[::-1], img[::-1, :]
    if x_ax[0] > x_ax[-1]:
        x_ax, img = x_ax[::-1], img[:, ::-1]
    return y_ax, x_ax, img


def reconstruct_classical(scene_name, method="linear", force=False):
    out_npz = DATA_DIR / f"{scene_name}_real_downscale_classical_{method}.npz"
    if out_npz.exists() and not force:
        return dict(np.load(out_npz, allow_pickle=True))

    g = build_20m_grid(scene_name)
    y_ax, x_ax, img_ax = _axes_ascending(g["y20"], g["x20"], g["img20"])

    interp = RegularGridInterpolator((y_ax, x_ax), img_ax, method=method,
                                      bounds_error=False, fill_value=None)
    yy10, xx10 = np.meshgrid(g["y10"], g["x10"], indexing="ij")
    rec10 = interp(np.stack([yy10.ravel(), xx10.ravel()], axis=-1)).reshape(g["img10"].shape)

    np.savez_compressed(
        out_npz, scene=scene_name, method=method,
        truth10=g["img10"].astype(np.float32),
        reconstructed10=rec10.astype(np.float32),
    )
    return dict(np.load(out_npz, allow_pickle=True))


## 9. Richardson-Lucy two-stage baseline

Deconvolve the 20 m observation with the same assumed Gaussian response used by the PSF-aware operator (`RECON_FWHM_M`), then bicubically upsample to the true 10 m grid -- the same two-stage structure used for this baseline throughout the paper.

In [ ]:
def reconstruct_richardson_lucy(scene_name, force=False):
    out_npz = DATA_DIR / f"{scene_name}_real_downscale_richardson_lucy.npz"
    if out_npz.exists() and not force:
        return dict(np.load(out_npz, allow_pickle=True))
    if richardson_lucy is None:
        raise ImportError("scikit-image richardson_lucy is not available.")

    g = build_20m_grid(scene_name)
    img20 = g["img20"]

    lo, hi = np.percentile(img20, [0.5, 99.5])
    hi = max(hi, lo + 1e-6)
    img01 = np.clip((img20 - lo) / (hi - lo), 0, 1)

    sigma_px = _fwhm_to_gauss_sigma_px(RECON_FWHM_M, pixel_size_m=PIXEL_SIZE_20M)
    radius = max(1, int(np.ceil(4 * sigma_px)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    psf = np.exp(-0.5 * (xx ** 2 + yy ** 2) / sigma_px ** 2)
    psf /= psf.sum()

    deconv01 = richardson_lucy(img01, psf, num_iter=RICHARDSON_LUCY_ITER, clip=False)
    deconv20 = deconv01 * (hi - lo) + lo

    y_ax, x_ax, img_ax = _axes_ascending(g["y20"], g["x20"], deconv20)
    interp = RegularGridInterpolator((y_ax, x_ax), img_ax, method="cubic",
                                      bounds_error=False, fill_value=None)
    yy10, xx10 = np.meshgrid(g["y10"], g["x10"], indexing="ij")
    rec10 = interp(np.stack([yy10.ravel(), xx10.ravel()], axis=-1)).reshape(g["img10"].shape)

    np.savez_compressed(
        out_npz, scene=scene_name,
        truth10=g["img10"].astype(np.float32),
        reconstructed10=rec10.astype(np.float32),
    )
    return dict(np.load(out_npz, allow_pickle=True))


## 10. Metrics and spectral utilities

All metrics are computed after `crop_border` removes a margin affected by the Gaussian blur's reflect-padding and by partially-supported HEALPix cells at the patch edge.

In [ ]:
def compute_metrics(truth, estimate, scene, method):
    t = crop_border(np.asarray(truth, dtype=np.float64))
    e = crop_border(np.asarray(estimate, dtype=np.float64))
    mask = np.isfinite(t) & np.isfinite(e)
    tv, ev = t[mask], e[mask]
    err = ev - tv

    rmse = float(np.sqrt(mean_squared_error(tv, ev)))
    mae = float(mean_absolute_error(tv, ev))
    bias = float(np.mean(err))
    corr = float(np.corrcoef(tv, ev)[0, 1]) if tv.size > 1 else np.nan
    dr = float(np.nanmax(t) - np.nanmin(t))
    psnr = (float("inf") if rmse == 0
            else float(20 * np.log10(dr / rmse)) if dr > 0 else np.nan)

    t_s = np.where(np.isfinite(t), t, np.nanmean(t))
    e_s = np.where(np.isfinite(e), e, np.nanmean(e))
    data_range = float(np.nanmax(t_s) - np.nanmin(t_s))
    ssim_val = float(ssim(t_s, e_s, data_range=data_range)) if data_range > 0 else np.nan

    return dict(scene=scene, method=method, rmse=rmse, mae=mae, bias=bias,
                corr=corr, psnr=psnr, ssim=ssim_val)


def prepare_for_psd(img, normalize=True, apodize=True):
    arr = np.asarray(img, dtype=np.float64)
    arr = np.where(np.isfinite(arr), arr, np.nanmean(arr))
    if normalize:
        p1, p99 = np.nanpercentile(arr, [1, 99])
        if p99 > p1:
            arr = np.clip((arr - p1) / (p99 - p1), 0, 1)
    arr = arr - np.mean(arr)
    if apodize:
        wy = np.hanning(arr.shape[0])
        wx = np.hanning(arr.shape[1])
        arr = arr * np.outer(wy, wx)
    return arr


def psd1d_numpy(img, pixel_size_m=PIXEL_SIZE_M):
    arr = prepare_for_psd(img)
    ft = np.fft.fftshift(np.fft.fft2(arr))
    psd2 = np.abs(ft) ** 2
    ny, nx = arr.shape
    yy, xx = np.indices((ny, nx))
    rr = np.sqrt((xx - nx // 2) ** 2 + (yy - ny // 2) ** 2).astype(np.int64)
    max_r = min(nx, ny) // 2
    psd_sum = np.bincount(rr.ravel(), weights=psd2.ravel(), minlength=max_r + 1)[:max_r + 1]
    psd_cnt = np.bincount(rr.ravel(), minlength=max_r + 1)[:max_r + 1]
    psd_mean = psd_sum / np.maximum(psd_cnt, 1)
    freq = np.arange(max_r + 1) / (max_r * 2 * pixel_size_m)
    return freq, psd_mean


## 11. Run everything

In [ ]:
def run_all(force=False):
    rows = []
    for scene in benchmark_coordinates:
        try:
            d = reconstruct_psf_aware(scene, force=force)
            rows.append(compute_metrics(d["truth10"], d["reconstructed10"], scene, "psf_aware"))
        except Exception as exc:
            print(f"[WARN] PSF-aware failed for {scene}: {exc}")

        for method in CLASSICAL_METHODS:
            try:
                d = reconstruct_classical(scene, method=method, force=force)
                rows.append(compute_metrics(d["truth10"], d["reconstructed10"], scene, f"classical_{method}"))
            except Exception as exc:
                print(f"[WARN] classical {method} failed for {scene}: {exc}")

        try:
            d = reconstruct_richardson_lucy(scene, force=force)
            rows.append(compute_metrics(d["truth10"], d["reconstructed10"], scene, "richardson_lucy"))
        except Exception as exc:
            print(f"[WARN] Richardson-Lucy failed for {scene}: {exc}")

    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "real_groundtruth_downscale_metrics.csv", index=False)
    return df


metrics_df = run_all(force=False)
metrics_df


### Results table (paper format)

In [ ]:
def format_table(df):
    methods = ["psf_aware"] + [f"classical_{m}" for m in CLASSICAL_METHODS] + ["richardson_lucy"]
    labels = {
        "psf_aware": "PSF-aware",
        "classical_nearest": "Nearest-neighbor",
        "classical_linear": "Bilinear",
        "classical_cubic": "Bicubic",
        "richardson_lucy": "Richardson-Lucy",
    }
    records = []
    for scene in benchmark_coordinates:
        row = {"Scene": scene}
        for m in methods:
            sub = df[(df["scene"] == scene) & (df["method"] == m)]
            if not sub.empty:
                row[f"{labels[m]} RMSE"] = sub.iloc[0]["rmse"]
                row[f"{labels[m]} SSIM"] = sub.iloc[0]["ssim"]
            else:
                row[f"{labels[m]} RMSE"] = np.nan
                row[f"{labels[m]} SSIM"] = np.nan
        records.append(row)
    table = pd.DataFrame(records)
    table.to_csv(TABLE_DIR / "real_groundtruth_downscale_table.csv", index=False)
    return table


format_table(metrics_df)


### Figure -- qualitative panels per scene

Each row is a scene; each column shows one method's reconstruction against the real 10 m truth and the synthetic 20 m observation actually fed to every method.

In [ ]:
def plot_panels(force=False):
    scenes = list(benchmark_coordinates.keys())
    col_labels = ["Truth (10 m)", "Degraded (20 m)", "PSF-aware", "Nearest", "Bilinear", "Bicubic", "Richardson-Lucy"]
    fig, axes = plt.subplots(len(scenes), len(col_labels), figsize=(2.6 * len(col_labels), 2.6 * len(scenes)))
    if len(scenes) == 1:
        axes = axes[np.newaxis, :]

    for r, scene in enumerate(scenes):
        psf_d = reconstruct_psf_aware(scene, force=force)
        truth = crop_border(psf_d["truth10"])
        vmin, vmax = np.nanpercentile(truth, [1, 99])

        panels = [truth, psf_d["degraded20"], crop_border(psf_d["reconstructed10"])]
        for method in CLASSICAL_METHODS:
            cl = reconstruct_classical(scene, method=method, force=force)
            panels.append(crop_border(cl["reconstructed10"]))
        rl = reconstruct_richardson_lucy(scene, force=force)
        panels.append(crop_border(rl["reconstructed10"]))

        for c, img in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(col_labels[c], fontsize=10)
            if c == 0:
                ax.set_ylabel(scene, fontsize=10)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "real_groundtruth_downscale_panels.pdf", dpi=200)
    plt.show()


plot_panels(force=False)


### Figure -- spectral comparison against the real 10 m ground truth

In [ ]:
def plot_spectra(force=False):
    scenes = list(benchmark_coordinates.keys())
    fig, axes = plt.subplots(1, len(scenes), figsize=(4.4 * len(scenes), 3.5))
    if len(scenes) == 1:
        axes = [axes]

    for ax, scene in zip(axes, scenes):
        psf_d = reconstruct_psf_aware(scene, force=force)
        truth = crop_border(psf_d["truth10"])
        freq, psd_truth = psd1d_numpy(truth)
        ax.loglog(freq[1:], psd_truth[1:], label="Truth (10 m)", color="black", lw=2)

        _, psd_psf = psd1d_numpy(crop_border(psf_d["reconstructed10"]))
        ax.loglog(freq[1:], psd_psf[1:], label="PSF-aware")

        for method in CLASSICAL_METHODS:
            cl = reconstruct_classical(scene, method=method, force=force)
            _, psd_cl = psd1d_numpy(crop_border(cl["reconstructed10"]))
            ax.loglog(freq[1:], psd_cl[1:], label=CLASSICAL_LABELS[method], alpha=0.7)

        rl = reconstruct_richardson_lucy(scene, force=force)
        _, psd_rl = psd1d_numpy(crop_border(rl["reconstructed10"]))
        ax.loglog(freq[1:], psd_rl[1:], label="Richardson-Lucy", alpha=0.7)

        ax.set_title(scene)
        ax.set_xlabel("spatial frequency (cycles/m)")
        if scene == scenes[0]:
            ax.set_ylabel("PSD")
            ax.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "real_groundtruth_downscale_spectra.pdf", dpi=200)
    plt.show()


plot_spectra(force=False)


### Figure -- RMSE summary

In [ ]:
def plot_rmse_summary(df):
    labels = {
        "psf_aware": "PSF-aware",
        "classical_nearest": "Nearest",
        "classical_linear": "Bilinear",
        "classical_cubic": "Bicubic",
        "richardson_lucy": "Richardson-Lucy",
    }
    methods = list(labels.keys())
    scenes = list(benchmark_coordinates.keys())

    fig, ax = plt.subplots(figsize=(8, 4.5))
    width = 0.15
    x = np.arange(len(scenes))
    for i, m in enumerate(methods):
        vals = [df[(df["scene"] == s) & (df["method"] == m)]["rmse"].mean() for s in scenes]
        ax.bar(x + (i - len(methods) / 2) * width, vals, width, label=labels[m])
    ax.set_xticks(x)
    ax.set_xticklabels(scenes)
    ax.set_ylabel("RMSE vs. real 10 m ground truth")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "real_groundtruth_downscale_rmse_summary.pdf", dpi=200)
    plt.show()


plot_rmse_summary(metrics_df)


## Notes and caveats

- **What this notebook does that the rest of the paper does not**: every
  number here is computed against a real, unsynthesized Sentinel-2 patch
  (`truth10`). Neither the round trip (no reference at all) nor the
  controlled Esri experiment (synthetic reference, matched degradation
  kernel) can make that claim.
- **What it does not establish**: there is no real 20 m Sentinel-2 B04 to
  compare `degraded20` against, so this is still a *simulated* degradation
  of real data, not two independently acquired sensors. The degradation
  pipeline (`DEGRADE_FWHM_M`, `BLOCK`) is a modelling choice, documented and
  configurable at the top of the notebook, not a measured instrument
  response -- treat the absolute RMSE numbers as method comparisons under
  one specific, reasonable degradation, not as a universal recovery-rate
  claim.
- **Deliberate PSF mismatch**: `RECON_FWHM_M` (assumed by the reconstruction
  operator and by Richardson-Lucy) is not fit to `DEGRADE_FWHM_M`/`BLOCK`
  (the true degradation), and the true degradation itself is not a pure
  Gaussian. This is intentional -- see the configuration cell -- and means
  any recovery seen here is not an artefact of the operator knowing the
  exact generating process.
- **Border cropping**: `BORDER_CROP_PX` excludes a margin from every metric
  to avoid conflating genuine reconstruction quality with edge effects from
  blur padding and partially-supported HEALPix cells at the patch boundary.
  It does not affect the reconstructions themselves, only the evaluation.
- **Extending this notebook**: `DEGRADE_FWHM_M`, `RECON_FWHM_M`, and `BLOCK`
  are all top-of-notebook constants. A natural follow-up is sweeping
  `RECON_FWHM_M` away from its current value to trace out a sensitivity
  curve against a *real* reference, the real-data analogue of the paper's
  synthetic +-50% PSF-mismatch arms.
